# 🎓 GenAI Capstone — Milestone 2: Agentic Assessment Design Assistant
## Step 1: Pedagogical Knowledge Base Ingestion (RAG Foundation)

**Prerequisite:** Milestone 1 classical ML predictor is in `colab.ipynb`.  
This notebook builds the LangGraph-based agentic system on top of it.

### Architecture Overview
```
Exam Questions → [LangGraph Agent] → Structured Report
                        ↓
              [RAG: ChromaDB + Pedagogy PDFs]
                        ↓  
              [LLM: Groq/Gemini Free Tier]
```

---

**Cell 2 — Install Dependencies**

In [1]:
# ── Step 1B: Install Milestone 2 Dependencies ──
!pip install -q \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-huggingface==0.0.3 \
    chromadb==0.5.3 \
    sentence-transformers==3.0.1 \
    pypdf==4.3.1

print("✅ All dependencies installed.")

✅ All dependencies installed.


**Cell 3 — Markdown**

---

## Step 1C: Ingest PDFs → Chunk → Embed → Store in ChromaDB


In [2]:
# ── Imports & Configuration ──
import os
import logging
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ── Paths ──
KNOWLEDGE_BASE_DIR = Path("pedagogy_knowledge_base")
CHROMA_DB_DIR      = Path("chroma_db")

# ── Embedding model (free, no API key needed) ──
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ── Chunk settings for educational documents ──
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 80

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

print("✅ Config loaded.")
print(f"   Knowledge Base : {KNOWLEDGE_BASE_DIR}")
print(f"   ChromaDB Path  : {CHROMA_DB_DIR}")
print(f"   Embedding Model: {EMBEDDING_MODEL}")


✅ Config loaded.
   Knowledge Base : pedagogy_knowledge_base
   ChromaDB Path  : chroma_db
   Embedding Model: sentence-transformers/all-MiniLM-L6-v2


In [3]:
# ── Load all PDFs from knowledge base ──
def load_pdfs(kb_dir: Path) -> list:
    pdf_files = list(kb_dir.glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(f"No PDFs found in '{kb_dir}'. Add your pedagogy PDFs first.")
    
    print(f"\n📂 Found {len(pdf_files)} PDF file(s):")
    all_docs = []
    for pdf_path in pdf_files:
        print(f"   Loading: {pdf_path.name} ...", end=" ")
        loader = PyPDFLoader(str(pdf_path))
        docs   = loader.load()
        print(f"→ {len(docs)} page(s)")
        all_docs.extend(docs)
    
    print(f"\n✅ Total raw pages loaded: {len(all_docs)}")
    return all_docs

raw_docs = load_pdfs(KNOWLEDGE_BASE_DIR)


📂 Found 2 PDF file(s):
   Loading: revised-blooms-taxonomy-action-verbs.pdf ... → 1 page(s)
   Loading: Assessment-Guide-February-2025.pdf ... → 14 page(s)

✅ Total raw pages loaded: 15


In [4]:
# ── Split into retrieval-optimised chunks ──
def chunk_documents(docs: list) -> list:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    return chunks

chunks = chunk_documents(raw_docs)

print(f"✅ Chunking complete.")
print(f"   Total chunks : {len(chunks)}")
print(f"   Chunk size   : {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars")
print(f"\n── Sample Chunk Preview ──")
print(f"Source : {chunks[0].metadata.get('source', 'N/A')}")
print(f"Content: {chunks[0].page_content[:200]}...")

✅ Chunking complete.
   Total chunks : 70
   Chunk size   : 512 chars | Overlap: 80 chars

── Sample Chunk Preview ──
Source : pedagogy_knowledge_base/revised-blooms-taxonomy-action-verbs.pdf
Content: REVISED Bloom’s Taxonomy  Action Verbs  
 
 
 
 
Definitions    I. Remembering   II. Understanding    III. Applying    IV. Analyzing    V. Evaluating    VI. Creating  
Bloom’s 
Definition   Exhibit me...


In [5]:
# ── Generate embeddings + persist to ChromaDB ──
# NOTE: First run downloads ~90MB model. Subsequent runs use cache.

print("⏳ Loading embedding model (downloading if first time)...")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print(f"✅ Embedding model loaded.")
print(f"\n⏳ Building ChromaDB at './{CHROMA_DB_DIR}' ...")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DB_DIR),
    collection_name="pedagogy_kb",
)

total_vectors = vector_store._collection.count()
print(f"✅ ChromaDB built successfully.")
print(f"   Vectors stored : {total_vectors}")
print(f"   Location       : ./{CHROMA_DB_DIR}/")

⏳ Loading embedding model (downloading if first time)...


/Users/atanuadhikari/Desktop/GenAICapstone2/.venv/lib/python3.14/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
12:02:46 [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


✅ Embedding model loaded.

⏳ Building ChromaDB at './chroma_db' ...


12:02:51 [INFO] Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
12:02:51 [ERROR] Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
12:02:51 [ERROR] Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ ChromaDB built successfully.
   Vectors stored : 140
   Location       : ./chroma_db/


In [6]:
# ── Sanity check: confirm retrieval works ──
test_queries = [
    "Bloom's taxonomy higher order thinking verbs",
    "assessment design best practices",
    "evaluating and creating level questions",
]

print("🔍 Running retrieval verification...\n")

for query in test_queries:
    results = vector_store.similarity_search(query, k=2)
    print(f"Query : '{query}'")
    if results:
        preview = results[0].page_content[:120].replace("\n", " ")
        print(f"Top-1 : {preview}...")
        print(f"Status: ✅ Retrieved\n")
    else:
        print(f"Status: ⚠️  No results — check PDF text is readable\n")

12:02:52 [ERROR] Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🔍 Running retrieval verification...

Query : 'Bloom's taxonomy higher order thinking verbs'
Top-1 : REVISED Bloom’s Taxonomy  Action Verbs           Definitions    I. Remembering   II. Understanding    III. Applying    I...
Status: ✅ Retrieved

Query : 'assessment design best practices'
Top-1 : 11 • Ensure Validity and Reliability:  Choose methods that accurately measure what they are intended to and  produce con...
Status: ✅ Retrieved

Query : 'evaluating and creating level questions'
Top-1 : 7 Try to keep the following questions in mind when developing learning outcomes:   • What are your goals? What knowledge...
Status: ✅ Retrieved



````markdown
## ✅ Step 1 Complete — RAG Foundation Ready

| Component | Status | Details |
|-----------|--------|---------|
| PDF Loader | ✅ Done | All PDFs from `pedagogy_knowledge_base/` loaded |
| Text Chunker | ✅ Done | `RecursiveCharacterTextSplitter` (512/80) |
| Embeddings | ✅ Done | `all-MiniLM-L6-v2` via HuggingFace (free) |
| Vector Store | ✅ Done | ChromaDB persisted at `./chroma_db/` |
| Retrieval Check | ✅ Done | Similarity search verified |

**Next → Step 2: LangGraph Agent Graph Setup**
````

In [7]:
!pip install -q \
    langgraph==0.2.16 \
    langchain-groq==0.1.9

print("✅ LangGraph + Groq installed.")

✅ LangGraph + Groq installed.


In [14]:
!pip install -q python-dotenv

from dotenv import load_dotenv
import os

load_dotenv()  # .env file automatically read karega

api_key = os.getenv("GROQ_API_KEY")

if api_key:
    print("✅ Groq API key loaded successfully.")
else:
    print("❌ Key not found. Check .env file.")

✅ Groq API key loaded successfully.


````markdown
## Step 2: LangGraph Agent Setup

### Agent Architecture
````
Input (Exam Questions)
        ↓
[Node 1: RAG Retriever]  ← ChromaDB (pedagogy_kb)
        ↓
[Node 2: LLM Analyser]   ← Groq (llama3-8b)
        ↓
[Node 3: Output Formatter]
        ↓
Structured Report (Summary, Gaps, Advice, Refs, Disclaimer)
````
````

Ye sirf markdown cell hai, koi code nahi — Colab/VS Code notebook mein **+ Markdown** click karke add karo. Cell 13 bol dena.

In [9]:
# ── Agent State Definition ──
from typing import TypedDict, List

class AssessmentState(TypedDict):
    """
    LangGraph agent ka shared state.
    Har node is state ko read/update karega.
    """
    exam_questions  : List[str]   # Input: list of exam questions
    retrieved_docs  : List[str]   # RAG se aaye pedagogy chunks
    analysis        : str         # LLM ka raw analysis
    summary         : str         # Quality & difficulty distribution
    gaps            : str         # Learning gaps identified
    advice          : str         # Improvement suggestions
    refs            : str         # Pedagogical references
    disclaimer      : str         # Ethical/educational notices

print("✅ AssessmentState defined.")
print("   Fields:", list(AssessmentState.__annotations__.keys()))

✅ AssessmentState defined.
   Fields: ['exam_questions', 'retrieved_docs', 'analysis', 'summary', 'gaps', 'advice', 'refs', 'disclaimer']


In [10]:
# ── LLM + Retriever Setup ──
from langchain_groq import ChatGroq
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# ── Load same embedding model used during ingestion ──
print("⏳ Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("✅ Embedding model loaded.")

# ── Load existing ChromaDB ──
print("⏳ Loading ChromaDB...")
vector_store = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings,
    collection_name="pedagogy_kb",
)
print(f"✅ ChromaDB loaded. Vectors: {vector_store._collection.count()}")

# ── Setup retriever ──
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},   # Top 4 relevant chunks fetch karega
)
print("✅ Retriever ready.")

# ── Setup Groq LLM ──
print("⏳ Connecting to Groq...")
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.3,           # Low temp = consistent structured output
    api_key=os.getenv("GROQ_API_KEY"),
)
print("✅ Groq LLM ready.")

12:02:53 [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


⏳ Loading embedding model...


/var/folders/ll/6g87r5c90ps4wkzl73sf6g100000gn/T/ipykernel_49413/2169622065.py:17: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vector_store = Chroma(
12:02:56 [ERROR] Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
12:02:56 [ERROR] Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Embedding model loaded.
⏳ Loading ChromaDB...
✅ ChromaDB loaded. Vectors: 140
✅ Retriever ready.
⏳ Connecting to Groq...
✅ Groq LLM ready.


In [11]:
# ── LangGraph Graph Assembly ──
from langgraph.graph import StateGraph, END

# ── Node 1: RAG Retriever ──
def rag_retriever_node(state: AssessmentState) -> AssessmentState:
    print("🔍 Node 1: Retrieving pedagogy docs...")
    
    # All questions ko ek query mein combine karo
    combined_query = " ".join(state["exam_questions"])
    
    docs = retriever.invoke(combined_query)
    retrieved_text = [doc.page_content for doc in docs]
    
    print(f"   Retrieved {len(retrieved_text)} chunks.")
    return {"retrieved_docs": retrieved_text}

# ── Node 2: LLM Analyser ──
def llm_analyser_node(state: AssessmentState) -> AssessmentState:
    print("🤖 Node 2: LLM analysing questions...")
    
    context = "\n\n".join(state["retrieved_docs"])
    questions = "\n".join([f"{i+1}. {q}" for i, q in enumerate(state["exam_questions"])])
    
    prompt = f"""
You are an expert educational assessment designer.

Using the pedagogy guidelines below, analyse the given exam questions.

PEDAGOGY GUIDELINES:
{context}

EXAM QUESTIONS:
{questions}

Provide your analysis in EXACTLY this format:

SUMMARY: (overall quality and difficulty distribution)

GAPS: (learning gaps or missing Bloom's taxonomy levels)

ADVICE: (specific improvements for each question)

REFS: (which pedagogy guidelines you referenced)

DISCLAIMER: (educational and ethical notices about this assessment)
"""
    response = llm.invoke(prompt)
    print("   LLM analysis complete.")
    return {"analysis": response.content}

# ── Node 3: Output Formatter ──
def output_formatter_node(state: AssessmentState) -> AssessmentState:
    print("📋 Node 3: Formatting structured output...")
    
    analysis = state["analysis"]
    
    def extract_section(text, section):
        try:
            start = text.index(f"{section}:") + len(f"{section}:")
            # Next section tak extract karo
            sections = ["SUMMARY", "GAPS", "ADVICE", "REFS", "DISCLAIMER"]
            next_sections = [s for s in sections if s != section]
            end = len(text)
            for ns in next_sections:
                try:
                    pos = text.index(f"{ns}:", start)
                    if pos < end:
                        end = pos
                except ValueError:
                    continue
            return text[start:end].strip()
        except ValueError:
            return "Not found in analysis."
    
    return {
        "summary"    : extract_section(analysis, "SUMMARY"),
        "gaps"       : extract_section(analysis, "GAPS"),
        "advice"     : extract_section(analysis, "ADVICE"),
        "refs"       : extract_section(analysis, "REFS"),
        "disclaimer" : extract_section(analysis, "DISCLAIMER"),
    }

# ── Build Graph ──
graph_builder = StateGraph(AssessmentState)

# Add nodes
graph_builder.add_node("rag_retriever",    rag_retriever_node)
graph_builder.add_node("llm_analyser",     llm_analyser_node)
graph_builder.add_node("output_formatter", output_formatter_node)

# Add edges (flow)
graph_builder.set_entry_point("rag_retriever")
graph_builder.add_edge("rag_retriever",    "llm_analyser")
graph_builder.add_edge("llm_analyser",     "output_formatter")
graph_builder.add_edge("output_formatter", END)

# Compile
agent = graph_builder.compile()

print("✅ LangGraph agent compiled successfully.")
print("   Flow: rag_retriever → llm_analyser → output_formatter → END")

✅ LangGraph agent compiled successfully.
   Flow: rag_retriever → llm_analyser → output_formatter → END


In [18]:
%pip install -q pandas
import pandas as pd

# ── Load dataset ──
df = pd.read_csv("final_project_dataset_v2.csv")

print("✅ Dataset loaded.")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
print(f"\n── Sample (first 3 rows) ──")
df.head()

Note: you may need to restart the kernel to use updated packages.
✅ Dataset loaded.
   Shape: (12300, 10)
   Columns: ['Question_ID', 'Question_Text', 'Subject_Domain', 'Topic_Subdomain', 'Bloom_Taxonomy', 'Exam_Type', 'Estimated_Duration_Mins', 'Student_Avg_Score', 'Pass_Rate', 'Difficulty_Level']

── Sample (first 3 rows) ──


,Question_ID,Question_Text,Subject_Domain,Topic_Subdomain,Bloom_Taxonomy,Exam_Type,Estimated_Duration_Mins,Student_Avg_Score,Pass_Rate,Difficulty_Level
0,QID_13455,Synthesize Relativity under extreme constraints.,Physics,Relativity,Evaluate,Quiz,11,2.4,49.0,Medium
1,QID_18063,State Networking briefly.,Computer Science,Networking,Remember,Final,5,4.4,87.9,Easy
2,QID_21615,Outline Mechanics for a specific case.,Physics,Mechanics,Apply,Midterm,14,3.5,70.0,Medium
3,QID_12677,State Database,Computer Science,Database,Remember,Quiz,17,4.6,91.7,Easy
4,QID_16794,Name Probability as defined in the textbook.,Mathematics,Probability,Remember,Midterm,9,4.3,86.5,Easy


In [19]:
# ── Sample 5 questions from dataset for testing ──
test_questions = df["Question_Text"].dropna().sample(5, random_state=42).tolist()

print("✅ Test questions selected:")
for i, q in enumerate(test_questions, 1):
    print(f"   {i}. {q[:100]}...")

✅ Test questions selected:
   1. Evaluate the limitations of Networking in distributed latency environments....
   2. Synthesize Database under extreme constraints....
   3. Define Mechanics briefly....
   4. Derive the formula for Database considering edge cases....
   5. Hypothesize Quantum Physics considering edge cases....


In [21]:
# ── Run LangGraph Agent ──
print("🚀 Starting Assessment Agent...\n")

initial_state = AssessmentState(
    exam_questions  = test_questions,
    retrieved_docs  = [],
    analysis        = "",
    summary         = "",
    gaps            = "",
    advice          = "",
    refs            = "",
    disclaimer      = "",
)

# Use a currently supported Groq model (old llama3-8b-8192 is decommissioned)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,
    api_key=os.getenv("GROQ_API_KEY"),
)

result = agent.invoke(initial_state)

print("\n✅ Agent run complete.")

🚀 Starting Assessment Agent...

🔍 Node 1: Retrieving pedagogy docs...
   Retrieved 4 chunks.
🤖 Node 2: LLM analysing questions...


12:10:00 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


   LLM analysis complete.
📋 Node 3: Formatting structured output...

✅ Agent run complete.


In [22]:
# ── Display Final Structured Report ──
print("=" * 60)
print("        INTELLIGENT ASSESSMENT ANALYSIS REPORT")
print("=" * 60)

print("\n📊 SUMMARY")
print("-" * 40)
print(result["summary"])

print("\n⚠️  GAPS")
print("-" * 40)
print(result["gaps"])

print("\n💡 ADVICE")
print("-" * 40)
print(result["advice"])

print("\n📚 REFS")
print("-" * 40)
print(result["refs"])

print("\n⚖️  DISCLAIMER")
print("-" * 40)
print(result["disclaimer"])

print("\n" + "=" * 60)
print("✅ Report generation complete.")

        INTELLIGENT ASSESSMENT ANALYSIS REPORT

📊 SUMMARY
----------------------------------------
** The exam questions cover a wide range of topics, including Networking, Database, Mechanics, and Quantum Physics. The difficulty level is moderate to challenging, requiring students to apply theoretical knowledge to practical problems. However, the questions lack a clear connection to real-world scenarios, making it difficult to assess students' ability to apply their knowledge in a practical context.

**

⚠️  GAPS
----------------------------------------
** The exam questions do not adequately assess students' ability to:

* Apply knowledge to real-world scenarios (Bloom's Taxonomy level 4: Application)
* Analyze complex systems and identify limitations (Bloom's Taxonomy level 5: Evaluation)
* Synthesize information from multiple sources to form a new understanding (Bloom's Taxonomy level 6: Synthesis)

**

💡 ADVICE
----------------------------------------
**

1. Evaluate the limitatio